# Exploratory Data Analysis — Drug Reviews Adverse Event Project

**Role 2 — Yosephine Tong**
Continues from `project_team_7.ipynb` (Role 1 — Jin-woo Hong), which covered dataset sourcing,
shape verification, missing value counts, summary statistics, and top conditions/drugs.

This notebook covers:
1. Dataset overview (types, nulls, sample rows)
2. Rating distribution
3. Review length distribution
4. Condition frequency
5. Temporal trends
6. Word frequency
7. Bigram and trigram analysis
8. VADER sentiment scoring vs patient ratings
9. LDA topic modeling on the focused subset (Depression + Anxiety)

**Before running:** execute `python src/preprocessing.py` from the repo root to generate
`data/processed/df_clean.csv` and `data/processed/df_focused.csv`.


## Setup

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from nltk.util import ngrams
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import nltk

warnings.filterwarnings("ignore")
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt_tab",    quiet=True)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 120

os.makedirs("outputs", exist_ok=True)
print("Setup complete.")


In [ ]:
# Load the cleaned outputs from preprocessing.py
# If these files don't exist, run: python src/preprocessing.py

df_clean   = pd.read_csv("data/processed/df_clean.csv",   parse_dates=["date"])
df_focused = pd.read_csv("data/processed/df_focused.csv", parse_dates=["date"])

print(f"df_clean   shape : {df_clean.shape}")
print(f"df_focused shape : {df_focused.shape}")
print(f"\nConditions in focused subset:")
print(df_focused["condition"].value_counts())


## 1. Dataset Overview

In [ ]:
print("Column types:")
print(df_clean.dtypes)
print("\nMissing values:")
print(df_clean.isnull().sum())
print("\nSample rows:")
df_clean[["drugName", "condition", "review_clean", "rating", "date"]].head(3)


## 2. Rating Distribution

Ratings run 1–10. We expect a skew toward higher values (mean ~7) since patients who
are satisfied are more likely to continue a medication and review it positively.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(df_clean["rating"], bins=10, range=(0.5, 10.5),
             color="steelblue", edgecolor="white", rwidth=0.85)
mean_rating = df_clean["rating"].mean()
axes[0].axvline(mean_rating, color="crimson", linestyle="--", linewidth=1.5,
                label=f"Mean: {mean_rating:.2f}")
axes[0].set_title("Rating Distribution — All Reviews")
axes[0].set_xlabel("Rating (1–10)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Boxplot by condition for focused subset
df_focused.boxplot(column="rating", by="condition", ax=axes[1], grid=False,
                   patch_artist=True)
axes[1].set_title("Rating by Condition — Focused Subset")
axes[1].set_xlabel("Condition")
axes[1].set_ylabel("Rating")
plt.suptitle("")

plt.tight_layout()
plt.savefig("outputs/rating_distribution.png", bbox_inches="tight")
plt.show()

print(df_clean["rating"].describe().round(2))


## 3. Review Length Distribution

Understanding review length matters for transformer model input limits. DistilBERT
has a max token length of 512; reviews longer than that will be truncated.

In [ ]:
df_clean["review_char_count"] = df_clean["review_clean"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count
axes[0].hist(df_clean["review_word_count"].clip(upper=500), bins=50,
             color="teal", edgecolor="white")
axes[0].set_title("Review Word Count Distribution")
axes[0].set_xlabel("Word Count (capped at 500)")
axes[0].set_ylabel("Number of Reviews")

# Character count
axes[1].hist(df_clean["review_char_count"].clip(upper=3000), bins=50,
             color="mediumorchid", edgecolor="white")
axes[1].set_title("Review Character Count Distribution")
axes[1].set_xlabel("Character Count (capped at 3,000)")
axes[1].set_ylabel("Number of Reviews")

plt.tight_layout()
plt.savefig("outputs/review_length_distribution.png", bbox_inches="tight")
plt.show()

print("Word count stats:")
print(df_clean["review_word_count"].describe().round(1))
print(f"\nReviews exceeding 512 words (DistilBERT limit): "
      f"{(df_clean['review_word_count'] > 512).sum():,} "
      f"({(df_clean['review_word_count'] > 512).mean()*100:.1f}%)")


## 4. Condition Frequency

In [ ]:
top_20 = df_clean["condition"].value_counts().head(20)

plt.figure(figsize=(10, 7))
bars = plt.barh(top_20.index[::-1], top_20.values[::-1], color="steelblue")
plt.xlabel("Number of Reviews")
plt.title("Top 20 Most Reviewed Conditions")

# Highlight our focus conditions
colors = ["crimson" if c in ["Depression", "Anxiety"] else "steelblue"
          for c in top_20.index[::-1]]
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.tight_layout()
plt.savefig("outputs/condition_frequency.png", bbox_inches="tight")
plt.show()

print("Depression and Anxiety are highlighted in red — our focus conditions.")


## 5. Temporal Trends

In [ ]:
df_clean["year"] = df_clean["date"].dt.year
reviews_per_year = df_clean.groupby("year").size().reset_index(name="count")
reviews_per_year = reviews_per_year[reviews_per_year["year"].between(2005, 2017)]

plt.figure(figsize=(10, 5))
plt.plot(reviews_per_year["year"], reviews_per_year["count"],
         marker="o", linewidth=2, color="steelblue")
plt.fill_between(reviews_per_year["year"], reviews_per_year["count"],
                 alpha=0.15, color="steelblue")
plt.title("Review Volume by Year")
plt.xlabel("Year")
plt.ylabel("Number of Reviews")
plt.xticks(reviews_per_year["year"].astype(int))
plt.tight_layout()
plt.savefig("outputs/temporal_trends.png", bbox_inches="tight")
plt.show()


## 6. Word Frequency

Using `review_processed` (stop words removed, lemmatized) from the focused subset so
results reflect meaningful content words rather than function words.

In [ ]:
all_tokens = []
for text in df_focused["review_processed"].dropna():
    all_tokens.extend(text.split())

word_freq = Counter(all_tokens).most_common(40)
words, counts = zip(*word_freq)

plt.figure(figsize=(12, 7))
plt.barh(list(words)[::-1], list(counts)[::-1], color="steelblue")
plt.xlabel("Frequency")
plt.title("Top 40 Words in Depression + Anxiety Reviews (stop words removed)")
plt.tight_layout()
plt.savefig("outputs/word_frequency.png", bbox_inches="tight")
plt.show()


## 7. Bigram and Trigram Analysis

In [ ]:
def get_ngram_freq(texts, n, top_k=20):
    all_ngrams = []
    for text in texts.dropna():
        tokens = text.split()
        all_ngrams.extend(ngrams(tokens, n))
    return Counter(all_ngrams).most_common(top_k)

bigram_freq  = get_ngram_freq(df_focused["review_processed"], 2)
trigram_freq = get_ngram_freq(df_focused["review_processed"], 3)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bigrams
bi_labels = [" ".join(g) for g, _ in bigram_freq]
bi_counts = [c for _, c in bigram_freq]
axes[0].barh(bi_labels[::-1], bi_counts[::-1], color="teal")
axes[0].set_title("Top 20 Bigrams — Focused Subset")
axes[0].set_xlabel("Frequency")

# Trigrams
tri_labels = [" ".join(g) for g, _ in trigram_freq]
tri_counts = [c for _, c in trigram_freq]
axes[1].barh(tri_labels[::-1], tri_counts[::-1], color="mediumorchid")
axes[1].set_title("Top 20 Trigrams — Focused Subset")
axes[1].set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("outputs/ngram_analysis.png", bbox_inches="tight")
plt.show()


## 8. VADER Sentiment Analysis

VADER scores each review on a compound scale from –1 (most negative) to +1 (most positive).
We compare it against the explicit patient rating to validate how well VADER tracks
stated satisfaction — and identify where they diverge.

RQ4 from the proposal: *How well does VADER sentiment scoring correlate with patient
ratings, and where do the two signals diverge?*

In [ ]:
sia = SentimentIntensityAnalyzer()

# Compute on focused subset (slower on full 215k — fine for EDA)
print("Computing VADER scores... (this takes ~1-2 minutes)")
df_focused = df_focused.copy()
df_focused["vader_score"] = df_focused["review_clean"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

# Classify into sentiment labels
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

df_focused["sentiment_label"] = df_focused["vader_score"].apply(label_sentiment)

print("Sentiment distribution:")
print(df_focused["sentiment_label"].value_counts())
print(f"\nCorrelation with rating: {df_focused['vader_score'].corr(df_focused['rating']):.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: vader_score vs rating
sample = df_focused.sample(min(3000, len(df_focused)), random_state=42)
axes[0].scatter(sample["rating"], sample["vader_score"],
                alpha=0.3, s=10, color="steelblue")
# Regression line
m, b = np.polyfit(sample["rating"], sample["vader_score"], 1)
x_line = np.linspace(1, 10, 100)
axes[0].plot(x_line, m * x_line + b, color="crimson", linewidth=2,
             label=f"r = {df_focused['vader_score'].corr(df_focused['rating']):.3f}")
axes[0].set_xlabel("Patient Rating (1–10)")
axes[0].set_ylabel("VADER Compound Score")
axes[0].set_title("VADER Score vs Patient Rating")
axes[0].legend()

# Sentiment distribution by condition
sentiment_counts = df_focused.groupby(["condition", "sentiment_label"]).size().unstack(fill_value=0)
sentiment_counts.plot(kind="bar", ax=axes[1], colormap="viridis", edgecolor="white")
axes[1].set_title("Sentiment Distribution by Condition")
axes[1].set_xlabel("Condition")
axes[1].set_ylabel("Number of Reviews")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Sentiment")

plt.tight_layout()
plt.savefig("outputs/vader_sentiment_analysis.png", bbox_inches="tight")
plt.show()


In [ ]:
# Where do VADER and ratings diverge?
# High rating but negative VADER — patient says they're satisfied but uses negative words
divergent_pos = df_focused[(df_focused["rating"] >= 8) & (df_focused["vader_score"] <= -0.2)]
# Low rating but positive VADER — patient is critical but phrases it positively
divergent_neg = df_focused[(df_focused["rating"] <= 3) & (df_focused["vader_score"] >= 0.2)]

print(f"High rating (≥8) but negative VADER (≤-0.2): {len(divergent_pos):,} reviews")
print(f"Low rating (≤3) but positive VADER (≥0.2):   {len(divergent_neg):,} reviews")
print("\nExample — high rating, negative VADER:")
print(divergent_pos["review_clean"].iloc[0][:300] if len(divergent_pos) > 0 else "None found")


## 9. LDA Topic Modeling

Latent Dirichlet Allocation on the focused subset (Depression + Anxiety) to surface
recurring adverse event themes. Uses `review_processed` (lemmatized, stop words removed)
as input to `CountVectorizer`.

In [ ]:
# Vectorize the focused subset
vectorizer = CountVectorizer(
    max_features=3000,
    min_df=5,          # ignore terms appearing in fewer than 5 reviews
    max_df=0.95,       # ignore terms appearing in more than 95% of reviews
    ngram_range=(1, 2) # include unigrams and bigrams
)

corpus = df_focused["review_processed"].dropna()
dtm    = vectorizer.fit_transform(corpus)
print(f"Document-term matrix shape: {dtm.shape}")


In [ ]:
lda = LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    max_iter=15,
    learning_method="online"
)
lda.fit(dtm)

feature_names = vectorizer.get_feature_names_out()

print("Top 10 words per topic:\n")
topic_labels = []
for i, topic_weights in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic_weights.argsort()[:-11:-1]]
    label = f"Topic {i}"
    topic_labels.append(label)
    print(f"{label}: {', '.join(top_words)}")


In [ ]:
# Assign each review its dominant topic
topic_dist = lda.transform(dtm)
dominant_topic = topic_dist.argmax(axis=1)

# Re-index corpus to align with dtm rows (dropna may have shifted index)
corpus_index = df_focused["review_processed"].dropna().index
df_topics = df_focused.loc[corpus_index].copy()
df_topics["topic_id"] = dominant_topic

# Topic distribution
topic_counts = df_topics["topic_id"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(topic_counts.index, topic_counts.values, color="steelblue", edgecolor="white")
plt.xlabel("Topic ID")
plt.ylabel("Number of Reviews")
plt.title("Review Count per LDA Topic — Focused Subset")
plt.xticks(topic_counts.index)
plt.tight_layout()
plt.savefig("outputs/lda_topic_distribution.png", bbox_inches="tight")
plt.show()

# Save enriched focused dataset with VADER + topic columns
df_topics.to_csv("data/processed/df_focused.csv", index=False)
print("Saved updated df_focused.csv with vader_score, sentiment_label, topic_id columns.")


## Summary

| Analysis | Output |
|---|---|
| Rating distribution | `outputs/rating_distribution.png` |
| Review length | `outputs/review_length_distribution.png` |
| Condition frequency | `outputs/condition_frequency.png` |
| Temporal trends | `outputs/temporal_trends.png` |
| Word frequency | `outputs/word_frequency.png` |
| Bigrams and trigrams | `outputs/ngram_analysis.png` |
| VADER vs ratings | `outputs/vader_sentiment_analysis.png` |
| LDA topic distribution | `outputs/lda_topic_distribution.png` |

**Handoff to Role 3 (Umang Khamar — NLP Engineer):**
Use `data/processed/df_focused.csv` which now includes:
- `review_clean` — for DistilBERT and BART input
- `review_processed` — for classical NLP methods
- `vader_score` — compound sentiment score
- `sentiment_label` — positive / neutral / negative
- `topic_id` — dominant LDA topic (0–7)

**Key finding for RQ4:** VADER correlates positively with patient ratings but diverges on
reviews where patients describe side effects in negative language while still rating the
drug highly overall (e.g., "it made me exhausted but it's working"). These divergences
are worth flagging for the summarizer's prompt design (Role 4).
